# 🎥 YouTube RAG — Chat With Any Video

Ask natural-language questions about a YouTube video and get grounded answers, generated from the video's own transcript using Retrieval-Augmented Generation (RAG). Supports multi-turn conversations, so follow-up questions ("what about after that?") work.

**Pipeline:** YouTube Transcript → Chunking → BM25 Keyword Retrieval (or direct timestamp lookup) → LLM (Groq, with conversation history) → Timestamped Answer

**Contents**
1. Install dependencies
2. Imports & logging
3. API key
4. Helpers — video ID parsing, transcript fetching, timestamps, tokenizing
5. Timestamp-aware questions
6. The RAG pipeline (with conversation memory)
7. Example usage
8. Optional — interactive Gradio chat demo

## 1. Install dependencies

In [1]:
%%capture
!pip install -q langchain-core langchain-groq langchain-text-splitters \
    youtube-transcript-api rank-bm25 python-dotenv gradio

## 2. Imports & logging

In [2]:
import os
import re
import logging
from getpass import getpass

from dotenv import load_dotenv
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
from rank_bm25 import BM25Okapi

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("yt_rag")

## 3. API key

The only key needed is Groq's (for the LLM).

Tries Colab/Kaggle secrets first (if running on one of those), then falls back to a `.env` file
(see `.env.example`), then finally prompts securely via `getpass` — so nothing ever gets
hardcoded or committed.

In [3]:
load_dotenv()

def _get_key(env_var: str, prompt: str) -> str:
    value = os.environ.get(env_var)
    if not value:
        value = getpass(prompt)
        os.environ[env_var] = value
    return value

GROQ_API_KEY = None
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    pass

if not GROQ_API_KEY:
    try:
        from kaggle_secrets import UserSecretsClient
        GROQ_API_KEY = UserSecretsClient().get_secret("GROQ_API_KEY")
    except Exception:
        pass

if not GROQ_API_KEY:
    GROQ_API_KEY = _get_key("GROQ_API_KEY", "Enter your Groq API key: ")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

## 4. Helpers — video ID parsing, transcript fetching, timestamps, tokenizing

In [4]:
YOUTUBE_ID_PATTERN = re.compile(
    r"(?:youtube\.com/(?:watch\?v=|embed/|shorts/)|youtu\.be/)([\w-]{11})"
)
TOKEN_PATTERN = re.compile(r"[a-z0-9]+")


def extract_video_id(url_or_id: str) -> str:
    """Accepts a full YouTube URL or a bare 11-character video ID and returns the video ID."""
    match = YOUTUBE_ID_PATTERN.search(url_or_id)
    if match:
        return match.group(1)
    if re.fullmatch(r"[\w-]{11}", url_or_id):
        return url_or_id
    raise ValueError(f"Could not extract a valid YouTube video ID from: {url_or_id!r}")


def fetch_transcript(video_id: str, languages=("en",)) -> list:
    """Fetches the transcript of a YouTube video as a list of {text, start, duration} dicts."""
    try:
        # youtube-transcript-api >= 1.0 (instance-based API)
        ytt_api = YouTubeTranscriptApi()
        fetched = ytt_api.fetch(video_id, languages=list(languages))
        return fetched.to_raw_data()
    except AttributeError:
        # youtube-transcript-api < 1.0 (old static method)
        try:
            return YouTubeTranscriptApi.get_transcript(video_id, languages=list(languages))
        except (TranscriptsDisabled, NoTranscriptFound):
            return []
        except Exception as exc:
            logger.error("Failed to fetch transcript for %s: %s", video_id, exc)
            return []
    except (TranscriptsDisabled, NoTranscriptFound):
        logger.error("No transcript/captions available for video %s", video_id)
    except Exception as exc:
        logger.error("Failed to fetch transcript for %s: %s", video_id, exc)
    return []


def format_timestamp(seconds: float) -> str:
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f"{h:02d}:{m:02d}:{s:02d}" if h else f"{m:02d}:{s:02d}"


def tokenize(text: str) -> list:
    """Simple lowercase word tokenizer used for BM25 (no ML model required)."""
    return TOKEN_PATTERN.findall(text.lower())

## 5. Timestamp-aware questions

BM25 is a *keyword* matcher — it has no concept of time, so a question like "what did they say
at 3 minutes 29 seconds" tokenizes into words (`what`, `did`, `minutes`, `seconds`...) that don't
meaningfully appear in the transcript text itself. BM25 ends up retrieving essentially arbitrary
chunks, and the LLM (correctly) says the answer isn't in them.

In [5]:
HH_MM_SS_PATTERN = re.compile(r"\b(\d{1,2}):(\d{2}):(\d{2})\b")
MM_SS_PATTERN = re.compile(r"\b(\d{1,2}):(\d{2})\b")
MINUTES_SECONDS_PATTERN = re.compile(
    r"(\d+)\s*(?:minutes?|mins?)\b(?:\s*(?:and)?\s*(\d+)\s*(?:seconds?|secs?)\b)?", re.I
)
SECONDS_ONLY_PATTERN = re.compile(r"(\d+)\s*(?:seconds?|secs?)\b", re.I)


def parse_timestamp_query(question: str):
    """Extracts a target time in seconds from phrasings like '3:29', '3 minutes 29 seconds',
    or '90 seconds'. Returns None if the question doesn't reference a specific timestamp."""
    m = HH_MM_SS_PATTERN.search(question)
    if m:
        h, mnt, s = map(int, m.groups())
        return h * 3600 + mnt * 60 + s

    m = MM_SS_PATTERN.search(question)
    if m:
        mnt, s = map(int, m.groups())
        return mnt * 60 + s

    m = MINUTES_SECONDS_PATTERN.search(question)
    if m:
        minutes = int(m.group(1))
        seconds = int(m.group(2)) if m.group(2) else 0
        return minutes * 60 + seconds

    m = SECONDS_ONLY_PATTERN.search(question)
    if m:
        return int(m.group(1))

    return None


def retrieve_by_timestamp(chunks, target_seconds, window=15, min_chunks=4):
    """Returns chunks within `window` seconds of the target time, in chronological order, so
    the LLM gets enough surrounding context to piece fragmented captions into a real answer.
    Falls back to the closest `min_chunks` if the window is too sparse (e.g. near the video's start/end)."""
    within_window = [c for c in chunks if abs(c.start - target_seconds) <= window]
    if len(within_window) >= min_chunks:
        return sorted(within_window, key=lambda c: c.start)
    closest = sorted(chunks, key=lambda c: abs(c.start - target_seconds))[:min_chunks]
    return sorted(closest, key=lambda c: c.start)

## 6. The RAG pipeline with conversation memory

Retrieval is BM25 by default, with the timestamp-lookup path from the previous section used
automatically when the question asks about a specific moment. The prompt now also carries the
last few turns of conversation, so follow-up questions like "what did they say right after
that?" have something to refer back to.

A video's transcript is only chunked
and indexed once per session, and the LLM chain is only built once per model.

In [6]:
from collections import namedtuple

TranscriptChunk = namedtuple("TranscriptChunk", ["text", "start"])

PROMPT = PromptTemplate(
    template=(
        "You are a helpful assistant answering questions about a YouTube video using ONLY the "
        "transcript excerpts below, plus the conversation so far.\n\n"
        "{history}"
        "Transcript excerpts (raw captions, so they may be choppy sentence fragments — piece them "
        "together to answer):\n{context}\n\n"
        "Question: {question}\n\n"
        "Instructions:\n"
        "- Base your answer only on the transcript excerpts; use the conversation history only to "
        "understand what a follow-up question is referring to, not as a source of new facts.\n"
        "- The excerpts are raw captions and will look fragmented — do your best to synthesize a "
        "clear answer from them rather than refusing just because they aren't full sentences.\n"
        "- Only say you don't know if the excerpts are genuinely unrelated to the question.\n"
        "- Be concise and cite approximate timestamps like [MM:SS] when relevant.\n\n"
        "Answer:"
    ),
    input_variables=["history", "context", "question"],
)

_index_cache = {}  # video_id -> (chunks, bm25_index)
_llm_cache = {}    # model_name -> chain


def build_chunks(transcript, chunk_size=1000, chunk_overlap=150):
    """Splits a transcript's segments into overlapping chunks, keeping each chunk's start time."""
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)

    chunks = []
    for entry in transcript:
        for piece in splitter.split_text(entry["text"]):
            chunks.append(TranscriptChunk(text=piece, start=entry["start"]))
    return chunks


def get_index(video_id, transcript, chunk_size=1000, chunk_overlap=150):
    """Builds (or reuses a cached) BM25 index for a video's transcript."""
    if video_id not in _index_cache:
        logger.info("Indexing transcript for video %s (%d segments)", video_id, len(transcript))
        chunks = build_chunks(transcript, chunk_size, chunk_overlap)
        tokenized = [tokenize(c.text) for c in chunks]
        bm25 = BM25Okapi(tokenized)
        _index_cache[video_id] = (chunks, bm25)
    return _index_cache[video_id]


def retrieve_chunks(chunks, bm25, question, top_k=4):
    """Returns the top-k most relevant chunks for a question via BM25 scoring."""
    scores = bm25.get_scores(tokenize(question))
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    relevant = [chunks[i] for i in top_indices if scores[i] > 0]
    return relevant or [chunks[i] for i in top_indices]


def format_chunks(chunks):
    """Formats retrieved chunks into a timestamped context block for the prompt."""
    lines = [f"[{format_timestamp(c.start)}] {c.text}" for c in chunks]
    return "\n\n".join(lines)


def format_history(chat_history, max_turns=4):
    """Formats prior (question, answer) turns into a short block for the prompt. Empty if no history."""
    if not chat_history:
        return ""
    recent = chat_history[-max_turns:]
    lines = [f"Q: {q}\nA: {a}" for q, a in recent]
    return "Conversation so far:\n" + "\n\n".join(lines) + "\n\n"


def get_chain(llm_model="openai/gpt-oss-120b"):
    """Builds (or reuses a cached) prompt | llm | parser chain for a given model."""
    if llm_model not in _llm_cache:
        llm = ChatGroq(model_name=llm_model, temperature=0.2)
        _llm_cache[llm_model] = PROMPT | llm | StrOutputParser()
    return _llm_cache[llm_model]


def ask_video(
    question: str,
    video: str,
    chat_history: list = None,
    llm_model: str = "openai/gpt-oss-120b",
    languages=("en",),
    chunk_size: int = 1000,
    chunk_overlap: int = 150,
    top_k: int = 4,
) -> str:
    """Answers a question about a YouTube video, given a URL or video ID.
    `chat_history` is an optional list of (question, answer) tuples from earlier in the conversation,
    used so follow-up questions can refer back to what was already asked."""
    video_id = extract_video_id(video)
    transcript = fetch_transcript(video_id, languages)
    if not transcript:
        return "Sorry, this video has no captions available, so I can't answer questions about it."

    chunks, bm25 = get_index(video_id, transcript, chunk_size, chunk_overlap)

    target_seconds = parse_timestamp_query(question)
    if target_seconds is not None:
        logger.info("Timestamp question detected (~%ds) — using direct time lookup instead of BM25", target_seconds)
        relevant = retrieve_by_timestamp(chunks, target_seconds)
    else:
        relevant = retrieve_chunks(chunks, bm25, question, top_k)

    context = format_chunks(relevant)
    history = format_history(chat_history)

    chain = get_chain(llm_model)
    return chain.invoke({"history": history, "context": context, "question": question})

## 7. Example usage

In [7]:
# Topic-based question -> BM25 retrieval
response_1 = ask_video(
    question="What is this video about?",
    video="https://www.youtube.com/watch?v=xAt1xcC6qfM",
)
print(response_1)

The clip is from a podcast that’s discussing a mix of topics – it’s “about” money and video content, touches on a death‑rate statistic, and then moves on to talk about AI. In short, the video is a podcast‑style conversation covering money, video, mortality data, and artificial intelligence. [24:01 – “because this whole podcast is about”]; [27:41 – “money and uh video”]; [24:53 – “death rate is about a third of what it…”]; [34:16 – “what about AI”].


In [8]:
# Timestamp-based question -> direct time lookup, bypassing BM25
response_2 = ask_video(
    question="What did they say at 3 minutes 29 seconds?",
    video="https://www.youtube.com/watch?v=xAt1xcC6qfM",
)
print(response_2)

At about 3 minutes 29 seconds they said, “hey you know what.” [03:28]


In [9]:
# Follow-up question -> passing chat_history lets the model resolve "that"
chat_history = [
    ("What did they say at 3 minutes 29 seconds?", response_2),
]
response_3 = ask_video(
    question="Can you say more about that?",
    video="https://www.youtube.com/watch?v=xAt1xcC6qfM",
    chat_history=chat_history,
)
print(response_3)

At that point the speaker is shifting the conversation toward what comes next. Just a few seconds earlier (around 3:04) they say they’re about to “tell you that there are more episodes” [03:04], and then, at roughly 3 minutes 29 seconds, they interject with the casual lead‑in “hey you know what.” [03:28] So the “hey you know what” line is basically a segue into the announcement that there are additional episodes coming up.


## 8. Optional — interactive Gradio chat demo

In [ ]:
import gradio as gr


def messages_to_pairs(history):
    """Converts Gradio's chat history into (question, answer) tuples.
    Handles both the older tuple-pairs format [[user_msg, bot_msg], ...] and the
    newer messages format [{'role': ..., 'content': ...}, ...]."""
    if not history:
        return []

    pairs = []
    if isinstance(history[0], dict):
        # newer "messages" format
        pending_question = None
        for m in history:
            if m["role"] == "user":
                pending_question = m["content"]
            elif m["role"] == "assistant" and pending_question is not None:
                pairs.append((pending_question, m["content"]))
                pending_question = None
    else:
        # older tuple-pairs format
        for turn in history:
            user_msg, bot_msg = turn[0], turn[1]
            if user_msg is not None and bot_msg is not None:
                pairs.append((user_msg, bot_msg))
    return pairs


def gradio_ask(message, history, video_url_or_id):
    if not video_url_or_id:
        return "Please enter a YouTube URL or video ID above first."
    try:
        chat_history = messages_to_pairs(history)
        return ask_video(message, video_url_or_id, chat_history=chat_history)
    except Exception as exc:
        return f"Error: {exc}"


demo = gr.ChatInterface(
    fn=gradio_ask,
    additional_inputs=[
        gr.Textbox(label="YouTube URL or Video ID", placeholder="https://youtu.be/dIb-DujRNEo"),
    ],
    title="🎥 YouTube RAG",
    description="Ask questions about any YouTube video using its transcript. Follow-up questions remember the conversation.",
)

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://818350976332e4e0aa.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
